<div style="text-align:center; padding: 60px 20px; border: 3px solid #1a1a2e; border-radius: 12px; background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); color: white; font-family: 'Segoe UI', sans-serif;">

<h1 style="font-size: 2.4em; letter-spacing: 2px; margin-bottom: 10px; color: #e94560;">
    📡 Simulador TR1
</h1>
<h2 style="font-size: 1.4em; font-weight: 300; color: #a8dadc; margin-bottom: 6px;">
    Teleinformática e Redes 1
</h2>
<hr style="border-color: #e94560; width: 40%; margin: 24px auto;">
<h3 style="font-size: 1.1em; font-weight: 400; color: #ccc; margin-bottom: 30px;">
    Simulação das Camadas Física e de Enlace de uma Rede de Comunicação
</h3>

<div style="background: rgba(255,255,255,0.05); border-radius: 8px; padding: 20px 40px; display: inline-block; text-align: left;">
<h3 style="color: #a8dadc; margin-bottom: 12px;">👥 Membros do Grupo</h3>
<p style="margin: 6px 0; font-size: 1.05em;">• <strong>Maria Eduarda Araujo Carvalho - 232023969</strong></p>
</div>

<br><br>
<p style="font-size: 0.9em; color: #888;">Universidade de Brasília &nbsp;|&nbsp; 2025</p>

</div>

---
## 1. Introdução

### Descrição do Problema

O objetivo deste trabalho é implementar um **simulador das camadas física e de enlace** de uma rede de comunicação digital, conforme especificado na disciplina de Teleinformática e Redes 1 (TR1).

O problema central consiste em transmitir uma mensagem de texto de um ponto (Transmissor — TX) a outro (Receptor — RX) passando por todas as etapas que uma comunicação real exigiria: codificação de bits, proteção contra erros, empacotamento em quadros, conversão em sinais analógicos e transporte pela rede.

### Visão Geral do Simulador

O simulador é composto por dois processos independentes que se comunicam via **TCP/IP na loopback** (`127.0.0.1:5000`):

| Processo | Papel | Interface |
|----------|-------|-----------|
| **Cliente** (`main_cliente.py`) | Transmissor (TX) | GTK3 — entrada de texto e exibição de gráficos |
| **Servidor** (`main_servidor.py`) | Receptor (RX) | GTK3 — log detalhado do processo de decodificação |

O pipeline completo, da mensagem ao texto recuperado, percorre as seguintes etapas:

```
[TX]  Texto → Bits → Hamming → Detecção de Erros → Enquadramento
      → Modulação Digital → Modulação por Portadora → Ruído Gaussiano → TCP

[RX]  TCP → Demodulação → Desenquadramento → Verificação de Erros
      → Correção Hamming → Bits → Texto
```

A cada transmissão o usuário pode escolher independentemente o **protocolo de enquadramento**, o **método de detecção de erros**, a **modulação digital** (banda-base) e a **modulação por portadora**, observando os efeitos de cada combinação nos gráficos exibidos pela interface.


---
## 2. Implementação

### 2.1 Estrutura do Projeto

```
tr1-final/
├── main_cliente.py          # Ponto de entrada do Transmissor (TX)
├── main_servidor.py         # Ponto de entrada do Receptor (RX)
├── auxiliar.py              # Conversões texto ↔ bits (UTF-8)
├── camada_fisica/
│   ├── modulacao.py         # NRZ-Polar, Manchester, Bipolar, ASK, FSK, QPSK, 16-QAM
│   ├── demodulacao.py       # Demoduladores correspondentes
│   └── ruido.py             # Ruído gaussiano aditivo
├── camada_enlace/
│   ├── enquadramento.py     # Contagem de Caracteres, Ins. de Bytes, Ins. de Bits
│   ├── deteccao_erros.py    # Paridade Par, Checksum 16-bit, CRC-32 (IEEE 802.3)
│   └── hamming.py           # Codificação e correção de erros de Hamming
├── cliente/
│   ├── transmitter.py       # Pipeline TX completo (classe Transmitter)
│   └── ui_cliente.py        # Interface GTK3 do TX
└── servidor/
    ├── receiver.py          # Servidor TCP + Pipeline RX (classe Receiver)
    └── ui_servidor.py       # Interface GTK3 do RX
```

A arquitetura separa completamente **lógica de protocolo** (módulos `camada_fisica` e `camada_enlace`) da **orquestração do pipeline** (classes `Transmitter` e `Receiver`) e da **interface gráfica** (`ui_cliente` e `ui_servidor`). Isso permite testar cada protocolo de forma isolada e facilita a manutenção.

---

### 2.2 Camada de Enlace

#### 2.2.1 Enquadramento

Três protocolos foram implementados, todos recebendo/retornando **strings de bits**:

| Protocolo | Transmissor | Receptor |
|-----------|------------|---------|
| **Contagem de Caracteres** | Prefixa cada quadro com 1 byte indicando o número de bytes de dado | Lê o cabeçalho e extrai exatamente esse número de bytes |
| **Inserção de Bytes** | Delimita quadros com FLAG (`0x26`) e escapa FLAG/ESC com `0x1B` (ESC) | Remove FLAGs e desfaz escapes |
| **Inserção de Bits** | Insere `'0'` após 5 bits `'1'` consecutivos; delimita com `01111110` | Remove o `'0'` stuffed após 5 uns e retira as flags |

**Decisão de projeto:** o tamanho máximo de quadro foi fixado em 6 bytes para os métodos baseados em bytes, valor suficiente para as mensagens de teste mas facilmente configurável.

#### 2.2.2 Detecção de Erros

| Protocolo | Overhead | Descrição |
|-----------|---------|-----------|
| **Paridade Par** | 1 bit | Bit `'1'` se a contagem de uns for ímpar; `'0'` caso contrário |
| **Checksum 16-bit** | 16 bits | Soma em complemento de 1 de palavras de 16 bits; complementa o resultado |
| **CRC-32 (IEEE 802.3)** | 32 bits | Polinômio `0x04C11DB7` com XOR inicial e final de `0xFFFFFFFF` |

**Decisão de projeto:** o CRC-32 é o padrão default do simulador por oferecer a maior capacidade de detecção. A implementação é puramente em Python (sem `binascii`), seguindo a divisão polinomial bit-a-bit para transparência didática.

#### 2.2.3 Correção de Erros — Código de Hamming

O Hamming é **sempre** aplicado (antes da detecção), com número de bits de paridade `r` calculado dinamicamente: o menor `r` tal que `2^r ≥ n + r + 1`, onde `n` é o comprimento da mensagem. Bits de paridade ocupam as posições potências de 2 (1, 2, 4, 8, …). O receptor calcula a síndrome e, se `pos_erro > 0`, inverte o bit correspondente antes de remover as posições de paridade.

---

### 2.3 Camada Física

#### 2.3.1 Modulações Digitais (Banda-Base)

Cada método produz um `numpy.ndarray` com **100 amostras por bit**:

| Método | Mapeamento |
|--------|-----------|
| **NRZ-Polar** | `1 → +1V`, `0 → −1V` |
| **Manchester** | `1 → [+V, −V]`, `0 → [−V, +V]` (200 amostras/bit) |
| **Bipolar (AMI)** | `0 → 0V`, `1 → alterna ±V` |

#### 2.3.2 Modulações por Portadora

| Método | Bits/símbolo | Portadora | Descrição |
|--------|-------------|-----------|-----------|
| **ASK** | 1 | 2 Hz | `1 → A·sin(2πft)`, `0 → 0` |
| **FSK** | 1 | f₁=2 Hz / f₀=1 Hz | Frequência varia com o bit |
| **QPSK** | 2 | 10 Hz | 4 pontos de constelação; decisão por mínima distância Euclidiana |
| **16-QAM** | 4 | 10 Hz | 16 pontos de constelação em grade 4×4; decisão por mínima distância |

A demodulação de QPSK e 16-QAM usa **correlação com portadoras I e Q** (produto interno) e buscca o símbolo de menor distância na constelação.

#### 2.3.3 Ruído

Um ruído gaussiano aditivo `n ~ N(0, σ²)` é somado ao sinal após a modulação, simulando imperfeições do canal. O desvio padrão `σ` é configurável pelo usuário na interface (`sigma_ruido`, padrão `0.05`).

---

### 2.4 Pipeline de Transmissão e Recepção

```
                        ┌─────────────────────────────────────┐
                        │         TRANSMISSOR (TX)            │
  Texto ──► texto_para_bits() ──► Hamming ──► Detecção Erros  │
                                                    │          │
             Modulação Digital ◄── Enquadramento ◄──┘          │
                    │                                          │
             Modulação Portadora ──► Ruído Gaussiano ──► TCP ──┤
                                                               │
                        └─────────────────────────────────────┘
                                         │ TCP/IP
                        ┌────────────────▼────────────────────┐
                        │         RECEPTOR (RX)               │
  ◄── bits_para_texto() ◄── Hamming ◄── Verif. Erros          │
                   Desenquadramento ◄── Demodulação ◄── TCP   │
                        └─────────────────────────────────────┘
```

**Protocolo TCP:** o TX serializa o sinal em JSON (arrays de tempo e sinal + metadados de protocolo) e prefixed com 4 bytes big-endian indicando o tamanho do payload. O RX usa `receber_exato()` para garantir recepção completa antes de processar.

---

### 2.5 Interface Gráfica

A UI foi construída com **GTK3 via PyGObject**. A janela do TX exibe:
- Campos para inserir texto, nome e configurar protocolos
- Dois gráficos Matplotlib embutidos: sinal digital e sinal de portadora (com e sem ruído)

A janela do RX exibe um terminal de log em tempo real, atualizado a cada mensagem recebida, com o resultado de cada etapa do pipeline.


---
## 3. Membros e Atividades

| Membro | Atividades Desenvolvidas |
|--------|-------------------------|
| **Maria** | Desenvolvimento integral do projeto: arquitetura do simulador, implementação das camadas física e de enlace, pipeline TX/RX, interface gráfica GTK3, testes e integração |

> O projeto foi desenvolvido individualmente. Todas as decisões de arquitetura, escolhas de algoritmos e código foram de responsabilidade da autora.


---
## 4. Conclusão

O simulador TR1 cumpre todos os requisitos da especificação: implementa três técnicas de enquadramento, três métodos de detecção de erros, correção via Hamming, três modulações digitais e quatro modulações por portadora, com transporte real via TCP e interface gráfica.

### Principais aprendizados

- **Camada física:** a implementação de QPSK e 16-QAM por correlação (produto interno com portadoras I e Q) tornou evidente como a demodulação coerente funciona na prática, algo que a teoria por si só não deixa tão concreto.
- **Enquadramento:** o *bit stuffing* exige cuidado com a flag de escape — uma sequência `011111110` legítima nos dados quebraria o enquadramento se não houvesse o stuffing do zero após cinco uns.
- **CRC-32:** a implementação bit-a-bit evidenciou a divisão polinomial; usar `binascii.crc32` seria mais eficiente, mas menos didático.
- **Integração TCP:** garantir o recebimento de exatamente N bytes (`receber_exato`) foi necessário porque o `socket.recv` não garante a entrega completa de um payload grande em uma única chamada.

### Principais dificuldades

- **Entendimento de conceitos de Camada Física:** Uma das principais dificuldades deste trabalho foi o próprio entendimento do conteúdo referente à camada física, que se mostrou bem abstrato e complicado de entender, e consequentemente aplicar na confecção trabalho.
- **Visualização dos gráficos:** Os gráficos também se mostraram difíceis de se projetar, uma vez que, ou ficavam completamente brancos, pretos ou não mostravam o gráfico referente à portadora.

---
## Apêndice — Demonstração dos Protocolos

As células abaixo executam exemplos de cada protocolo diretamente no notebook, sem precisar da interface gráfica.


In [ ]:
import sys
import zipfile, pathlib, tempfile, shutil

# Extrai o projeto para um diretório temporário e adiciona ao path
_ZIP = pathlib.Path("/mnt/user-data/uploads/tr1-final-master.zip")
_TMP = pathlib.Path(tempfile.mkdtemp())
with zipfile.ZipFile(_ZIP) as z:
    z.extractall(_TMP)

_ROOT = _TMP / "tr1-final-master"
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

print("Projeto carregado em:", _ROOT)


In [ ]:
from auxiliar import texto_para_bits, bits_para_texto
from camada_enlace.hamming import transmissor_hamming, receptor_hamming
from camada_enlace.deteccao_erros import transmissor_crc, receptor_crc
from camada_enlace.enquadramento import (
    transmissor_contagem, receptor_contagem,
    transmissor_insercao_bytes, receptor_insercao_bytes,
    transmissor_insercao_bits, receptor_insercao_bits,
)

mensagem = "TR1"
bits = texto_para_bits(mensagem)
print(f"Texto              : {mensagem}")
print(f"Bits               : {bits}")

bits_h = transmissor_hamming(bits)
print(f"\nApós Hamming       : {bits_h}")

bits_crc = transmissor_crc(bits_h)
print(f"Após CRC-32        : {bits_crc[:40]}… ({len(bits_crc)} bits)")

bits_eq = transmissor_contagem(bits_crc)
print(f"Após Enquadramento : {bits_eq[:48]}… ({len(bits_eq)} bits)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from camada_fisica.modulacao import nrz_polar, manchester, bipolar, ask, fsk

sample_bits = [1, 0, 1, 1, 0, 0, 1, 0]

fig, axes = plt.subplots(5, 1, figsize=(14, 12))
fig.suptitle("Sinais gerados pelos protocolos de modulação", fontsize=14, fontweight='bold')

# NRZ-Polar
s = nrz_polar(sample_bits)
t = np.arange(len(s))
axes[0].plot(t, s, color='#e94560', linewidth=1.2)
axes[0].set_title("NRZ-Polar")
axes[0].set_ylabel("Amplitude")

# Manchester
s = manchester(sample_bits)
t = np.arange(len(s))
axes[1].plot(t, s, color='#0f3460', linewidth=1.2)
axes[1].set_title("Manchester")
axes[1].set_ylabel("Amplitude")

# Bipolar (AMI)
s = bipolar(sample_bits)
t = np.arange(len(s))
axes[2].plot(t, s, color='#533483', linewidth=1.2)
axes[2].set_title("Bipolar (AMI)")
axes[2].set_ylabel("Amplitude")

# ASK
t_ask, s_ask = ask(sample_bits)
axes[3].plot(t_ask, s_ask, color='#06d6a0', linewidth=1.0)
axes[3].set_title("ASK (portadora f=2 Hz)")
axes[3].set_ylabel("Amplitude")

# FSK
t_fsk, s_fsk = fsk(sample_bits)
axes[4].plot(t_fsk, s_fsk, color='#ff6b35', linewidth=1.0)
axes[4].set_title("FSK (f1=2 Hz / f0=1 Hz)")
axes[4].set_ylabel("Amplitude")
axes[4].set_xlabel("Amostras")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()
print("Bits usados:", sample_bits)


In [ ]:
from camada_fisica.modulacao import qpsk, qam16
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
fig.suptitle("Modulações por Portadora — QPSK e 16-QAM", fontsize=13, fontweight='bold')

t_q, s_q = qpsk(sample_bits)
axes[0].plot(t_q, s_q, color='#e94560', linewidth=0.9)
axes[0].set_title("QPSK (2 bits/símbolo, portadora 10 Hz)")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)

t_16, s_16 = qam16(sample_bits)
axes[1].plot(t_16, s_16, color='#0f3460', linewidth=0.9)
axes[1].set_title("16-QAM (4 bits/símbolo, portadora 10 Hz)")
axes[1].set_ylabel("Amplitude")
axes[1].set_xlabel("Tempo (s)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Pipeline completo TX → RX (sem rede)
from cliente.transmitter import Transmitter
from camada_fisica.demodulacao import demod_ask

tx = Transmitter(
    mod_digital='NRZ-Polar',
    mod_portadora='ASK',
    enquadramento='Contagem de Caracteres',
    deteccao='CRC',
    sigma_ruido=0.0,   # sem ruído para demonstração limpa
)

resultado = tx.processar("Olá, TR1!")
t, sinal = resultado['sinal_portadora']

# Demodula manualmente
from camada_fisica.demodulacao import demod_ask
from camada_enlace.enquadramento import receptor_contagem
from camada_enlace.deteccao_erros import receptor_crc
from camada_enlace.hamming import receptor_hamming
from auxiliar import bits_para_texto

bits_demod = ''.join(map(str, demod_ask(t, sinal)))
bits_desenq = receptor_contagem(bits_demod)
_, bits_sem_crc = receptor_crc(bits_desenq)
bits_corr, pos = receptor_hamming(bits_sem_crc)
texto_rec = bits_para_texto(bits_corr)

print(f"Mensagem original  : {resultado['texto']}")
print(f"Bits brutos        : {resultado['bits_brutos']}")
print(f"Pos. erro Hamming  : {pos} (0 = sem erro)")
print(f"Mensagem recuperada: {texto_rec}")
assert resultado['texto'] == texto_rec, "ERRO: mensagens diferem!"
print("\n✓ Pipeline TX→RX validado com sucesso!")
